In [2]:
from tqdm import tqdm
import numpy as np
import json
import time
import pandas as pd
from neo4j import GraphDatabase, basic_auth

In [3]:
with open('../data/open_alex_graph.json') as f:
    oalex_graph_dict = json.load(f)

In [4]:
df = pd.DataFrame({
    'src': list(oalex_graph_dict.keys()),
    'dest': list(oalex_graph_dict.values())
}
)

In [5]:
df['dest'] = list(map(';'.join, df['dest']))

In [6]:
df.head()

,src,dest
0,https://openalex.org/W1999172455,https://openalex.org/W67955181;https://openale...
1,https://openalex.org/W2066928792,https://openalex.org/W1971337798;https://opena...
2,https://openalex.org/W2133314499,https://openalex.org/W1526226205;https://opena...
3,https://openalex.org/W2009331888,https://openalex.org/W93066099;https://openale...
4,https://openalex.org/W2150498921,https://openalex.org/W140019872;https://openal...


In [50]:
class Neo4jConnection:
    
    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None
        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
            self.__driver.verify_connectivity()
        except Exception as e:
            print("Failed to create the driver:", e)
        
    def close(self):
        if self.__driver is not None:
            self.__driver.close()
        
    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None
        try: 
            session = self.__driver.session(database=db) if db is not None else self.__driver.session() 
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally: 
            if session is not None:
                session.close()
        return response


conn = Neo4jConnection(uri="bolt://44.201.234.236:7687", 
                       user="neo4j",              
                       pwd="misleads-integers-realinement")

In [51]:
conn.query('CREATE CONSTRAINT papers IF NOT EXISTS FOR (p:Paper)     REQUIRE p.id IS UNIQUE')

[]

In [52]:
def insert_data(query, rows, batch_size = 10000):
    # Function to handle the updating the Neo4j database in batch mode.
    
    total = 0
    batch = 0
    start = time.time()
    result = None
    
    while batch * batch_size < len(rows):

        res = conn.query(query, 
                         parameters = {'rows': rows[batch*batch_size:(batch+1)*batch_size].to_dict('records')})
        total += res[0]['total']
        batch += 1
        result = {"total":total, 
                  "batches":batch, 
                  "time":time.time()-start}
        print(result)
        
    return result

In [57]:
def add_papers(rows, batch_size=5000):
   # Adds paper nodes and (:Author)--(:Paper) and 
   # (:Paper)--(:Category) relationships to the Neo4j graph as a 
   # batch job.
 
   query = '''
   UNWIND $rows as row
   MERGE (p:Paper {id:row.src})
 
   // connect categories
   WITH row, p
   UNWIND split(row.dest, ':') AS dest
   MERGE (p1:Paper {id: dest})
   MERGE (p)-[:CITES]->(p1)
   
   RETURN count(distinct p) as total
   '''
 
   return insert_data(query, rows, batch_size)

In [58]:
add_papers(df, batch_size=5000)

Failed to write data to connection IPv4Address(('44.201.234.236', 7687)) (IPv4Address(('44.201.234.236', 7687)))


KeyboardInterrupt: 

In [14]:
df.to_csv('openAlexCitations.csv')

In [67]:
for i, df_sub in enumerate(tqdm(np.array_split(df, 40))):
    df_sub.to_csv(f'../openAlexData/openAlexCitations{i}.csv')

100%|███████████████████████████████████████████| 40/40 [00:33<00:00,  1.19it/s]


In [8]:
df.to_csv(f'../data/openAlexCitations.csv')

In [12]:
check_csv = pd.read_csv('openAlexCitations.csv')

In [13]:
check_csv.head()

,Unnamed: 0,src,dest
0,0,https://openalex.org/W1999172455,"['https://openalex.org/W67955181', 'https://op..."
1,1,https://openalex.org/W2066928792,"['https://openalex.org/W1971337798', 'https://..."
2,2,https://openalex.org/W2133314499,"['https://openalex.org/W1526226205', 'https://..."
3,3,https://openalex.org/W2009331888,"['https://openalex.org/W93066099', 'https://op..."
4,4,https://openalex.org/W2150498921,"['https://openalex.org/W140019872', 'https://o..."
